In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
builder = SparkSession.builder.appName("Local_Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.caseSensitive", "true")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/10 18:52:02 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/10 18:52:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d0dd5b86-7ce7-4b09-8d7c-c3b7cb11b763;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in 

### Reading data

In [4]:
df = spark.read \
    .format("json") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/posts_results.jsonl")

In [5]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

In [6]:
df.show(truncate=False,n=10)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
df.printSchema()

root
 |-- author: struct (nullable = true)
 |    |-- associated: struct (nullable = true)
 |    |    |-- activitySubscription: struct (nullable = true)
 |    |    |    |-- allowSubscriptions: string (nullable = true)
 |    |    |-- chat: struct (nullable = true)
 |    |    |    |-- allowIncoming: string (nullable = true)
 |    |    |-- germ: struct (nullable = true)
 |    |    |    |-- messageMeUrl: string (nullable = true)
 |    |    |    |-- showButtonTo: string (nullable = true)
 |    |    |-- labeler: boolean (nullable = true)
 |    |-- avatar: string (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- did: string (nullable = true)
 |    |-- displayName: string (nullable = true)
 |    |-- handle: string (nullable = true)
 |    |-- labels: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- cid: string (nullable = true)
 |    |    |    |-- cts: string (nullable = true)
 |    |    |    |-- exp: string (nullable = true

In [8]:
df.select("likeCount").show(n=10,truncate=False)

+---------+
|likeCount|
+---------+
|2        |
|698      |
|41       |
|9        |
|35       |
|37       |
|58       |
|281      |
|320      |
|0        |
+---------+
only showing top 10 rows


In [9]:
from pyspark.sql.functions import *

In [10]:
df=df.withColumn("splitted_text",array_distinct(split(lower(col("record.text")),r"\s+")))

In [11]:
# df.show(n=20,truncate=False)

In [12]:
df=df.withColumn("exploded_text",explode(col("splitted_text")))

In [13]:
df.show(n=20,truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
numPosts=df.groupBy("exploded_text").agg(count("*").alias("count")).sort(desc("count"))

In [18]:
numPosts.show(n=100,truncate=False)

+-------------+------+
|exploded_text|count |
+-------------+------+
|the          |889169|
|to           |704081|
|a            |676189|
|and          |592265|
|of           |537603|
|i            |503280|
|in           |454349|
|is           |435372|
|for          |373034|
|on           |311230|
|this         |308386|
|that         |272305|
|my           |269845|
|it           |243833|
|with         |230244|
|you          |226163|
|but          |190610|
|be           |184941|
|have         |173095|
|at           |169326|
|so           |167595|
|are          |166342|
|not          |152849|
|just         |149315|
|from         |143496|
|was          |141407|
|me           |140990|
|like         |137855|
|as           |126847|
|             |126185|
|if           |125166|
|all          |120675|
|about        |120510|
|by           |112853|
|an           |110397|
|we           |108623|
|what         |107627|
|they         |105415|
|out          |101347|
|one          |100901|
|do        

In [19]:
numHashtags=numPosts.filter(col("exploded_text").startswith("#"))

In [20]:
numHashtags.show(n=100,truncate=False)

+-------------------+-----+
|exploded_text      |count|
+-------------------+-----+
|#art               |16435|
|#nsfw              |13731|
|#photography       |8223 |
|#oc                |6797 |
|#furry             |6248 |
|#gay               |5168 |
|#furryart          |4718 |
|#iran              |4409 |
|#digitalart        |4353 |
|#musicchallenge    |4141 |
|#trump             |3919 |
|#porn              |3837 |
|#nowplaying        |3661 |
|#ai                |3422 |
|#news              |3395 |
|#ass               |3364 |
|#fanart            |3075 |
|#booksky           |3018 |
|#nature            |3017 |
|#vtuber            |3010 |
|#birds             |2969 |
|#musicsky          |2959 |
|#nude              |2957 |
|#nsfwsky           |2791 |
|#ad                |2572 |
|#sexy              |2571 |
|#horny             |2423 |
|#goon              |2324 |
|#milf              |2317 |
|#aewdynamite       |2304 |
|#crypto            |2268 |
|#cock              |2262 |
|#ocsky             